# 06 - Simulação Realista com Gestão de Banca

Simulação das estratégias **1-2-4** e **1-2** com regras reais:
- Banca inicial: R$500,00
- Saque de R$100 ao atingir R$600 (lucro de R$100)
- Reposição de R$500 em caso de banca zerada
- Apenas horários verdes (melhor win rate)
- Trigger: 6 LOWs consecutivos | Target: 2.0x

In [ ]:
import sys
sys.path.insert(0, '.')
from config_analysis import *

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import plotly.io as pio
pio.templates.default = 'plotly_dark'

CYAN, MAGENTA, GREEN, RED, YELLOW = '#00f0ff', '#ff00ff', '#00ff88', '#ff3366', '#ffff00'

df = load_processed_data()
if df is None:
    raise FileNotFoundError('Execute o notebook 01 primeiro!')

print(f'Dataset: {len(df):,} registros')
print(f'Período: {df["date"].min().strftime("%Y-%m-%d")} a {df["date"].max().strftime("%Y-%m-%d")}')

## 1. Configuração

In [ ]:
# === PARÂMETROS ===
BANKROLL_INICIAL = 500.0
META_SAQUE = 100.0          # sacar quando lucro >= 100
TRIGGER = 6                 # entrar após 6 LOWs consecutivos
TARGET = 2.0                # multiplicador mínimo para win

# Estratégias
STRATEGIES = {
    '1-2-4': {'pattern': [1, 2, 4], 'divisor': 7},
    '1-2':   {'pattern': [1, 2],    'divisor': 3},
}

# === HORÁRIOS VERDES (do heatmap Hora x Dia do notebook 02) ===
# Calcular % LOW por (dia_semana, hora) e selecionar os slots verdes
heatmap = df.groupby(['dia_semana', 'hora'])['is_low'].mean() * 100
global_mean = df['is_low'].mean() * 100

# Slots verdes = (dia, hora) com % LOW abaixo da média global
# Quanto menor o % LOW, melhor para a estratégia (mais chance de HIGH após streak)
GREEN_SLOTS = set()
for (dow, hour), pct in heatmap.items():
    if pct < global_mean:
        GREEN_SLOTS.add((dow, hour))

# Visualizar quais slots são verdes
dias = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']
print(f'Média global de LOWs: {global_mean:.2f}%')
print(f'Slots verdes (% LOW < média): {len(GREEN_SLOTS)} de 168 ({len(GREEN_SLOTS)/168*100:.0f}%)\n')

print(f'{"":>5}', end='')
for h in range(24):
    print(f'{h:>5}', end='')
print()
print('-' * (5 + 24*5))
for d in range(7):
    print(f'{dias[d]:>5}', end='')
    for h in range(24):
        pct = heatmap.get((d, h), 0)
        marker = '  ✓  ' if (d, h) in GREEN_SLOTS else '  ·  '
        print(marker, end='')
    print()

print(f'\n✓ = Slot verde (opera)  · = Slot vermelho (não opera)')
print(f'\nBanca inicial:  R${BANKROLL_INICIAL:,.2f}')
print(f'Meta de saque:  R${META_SAQUE:,.2f} de lucro')
print(f'Trigger:        {TRIGGER} LOWs | Target: {TARGET}x')
print(f'\nEstratégias:')
for name, cfg in STRATEGIES.items():
    print(f'  {name}: pattern={cfg["pattern"]}, unit=banca/{cfg["divisor"]}')

## 2. Preparação dos Dados

In [ ]:
# Calcular streaks no dataset COMPLETO (o bot monitora 24h)
mults = df['multiplicador'].values
is_low = (mults < TARGET).astype(int)
hours = df['hora'].values
weekdays = df['dia_semana'].values
dates = df['date'].values

n = len(mults)
streaks = np.zeros(n, dtype=int)
for i in range(1, n):
    if is_low[i] == 1:
        streaks[i] = streaks[i - 1] + 1

# Máscara de slots verdes (dia_semana, hora)
green_mask = np.array([(weekdays[i], hours[i]) in GREEN_SLOTS for i in range(n)])

print(f'Rounds totais:           {n:,}')
print(f'Rounds em slots verdes:  {green_mask.sum():,} ({green_mask.mean()*100:.1f}%)')
print(f'Sinais (streak>={TRIGGER}) totais:   {(streaks >= TRIGGER).sum():,}')
print(f'Sinais em slots verdes:            {((streaks >= TRIGGER) & green_mask).sum():,}')

## 3. Motor de Simulação Realista

In [ ]:
def simulate_strategy(
    mults, streaks, hours, weekdays, dates, green_slots,
    pattern, divisor, trigger, target,
    bankroll_inicial, meta_saque,
):
    """
    Simulação realista com gestão de banca.
    
    - Streaks calculadas no dataset completo (bot monitora 24h)
    - Apostas APENAS em slots verdes (dia_semana, hora)
    - Saque ao atingir lucro de meta_saque
    - Reposição integral ao zerar a banca
    """
    n = len(mults)
    bankroll = bankroll_inicial
    total_deposited = bankroll_inicial
    total_withdrawn = 0.0
    n_deposits = 1
    n_withdrawals = 0
    
    # Tracking
    equity_curve = []
    bankroll_curve = []
    trades = []
    withdrawals_log = []
    deposits_log = []
    
    # Unit é calculado no início de cada ciclo de martingale
    cycle_unit = None
    
    for i in range(n - 1):
        current_streak = streaks[i]
        
        # Só opera em slots verdes (dia_semana, hora)
        if (weekdays[i], hours[i]) not in green_slots:
            continue
        
        # Sem saldo, repor
        if bankroll <= 0:
            bankroll = bankroll_inicial
            total_deposited += bankroll_inicial
            n_deposits += 1
            deposits_log.append({'index': i, 'date': dates[i], 'amount': bankroll_inicial})
        
        if current_streak < trigger:
            cycle_unit = None
            continue
        
        position = current_streak - trigger  # 0, 1, 2...
        
        if position >= len(pattern):
            continue  # ciclo esgotado
        
        # Novo ciclo: calcular unit com banca atual
        if position == 0:
            cycle_unit = bankroll / divisor
        
        if cycle_unit is None or cycle_unit <= 0:
            continue
        
        bet = cycle_unit * pattern[position]
        bet = min(bet, bankroll)
        
        if bet < 0.01:
            continue
        
        # Resultado do PRÓXIMO round
        next_mult = mults[i + 1]
        
        if next_mult >= target:
            pnl = bet * (target - 1)
            hit = True
        else:
            pnl = -bet
            hit = False
        
        bankroll += pnl
        bankroll = round(bankroll, 2)
        
        net_equity = total_withdrawn + bankroll - total_deposited
        
        trades.append({
            'index': i,
            'date': dates[i],
            'hour': hours[i],
            'weekday': weekdays[i],
            'streak': current_streak,
            'position': position,
            'bet': round(bet, 2),
            'next_mult': next_mult,
            'hit': hit,
            'pnl': round(pnl, 2),
            'bankroll': bankroll,
            'net_equity': net_equity,
        })
        
        equity_curve.append(net_equity)
        bankroll_curve.append(bankroll)
        
        # Verificar meta de saque
        if bankroll >= bankroll_inicial + meta_saque:
            withdraw = bankroll - bankroll_inicial
            total_withdrawn += withdraw
            n_withdrawals += 1
            withdrawals_log.append({
                'index': i, 'date': dates[i],
                'amount': withdraw, 'total': total_withdrawn,
            })
            bankroll = bankroll_inicial
        
        # Verificar bust
        if bankroll <= 0:
            bankroll = bankroll_inicial
            total_deposited += bankroll_inicial
            n_deposits += 1
            deposits_log.append({'index': i, 'date': dates[i], 'amount': bankroll_inicial})
    
    trades_df = pd.DataFrame(trades)
    wins = trades_df['hit'].sum() if len(trades_df) > 0 else 0
    total_trades = len(trades_df)
    
    return {
        'trades_df': trades_df,
        'equity_curve': equity_curve,
        'bankroll_curve': bankroll_curve,
        'total_trades': total_trades,
        'wins': wins,
        'losses': total_trades - wins,
        'win_rate': wins / total_trades if total_trades > 0 else 0,
        'final_bankroll': bankroll,
        'total_deposited': total_deposited,
        'total_withdrawn': total_withdrawn,
        'n_deposits': n_deposits,
        'n_withdrawals': n_withdrawals,
        'net_profit': total_withdrawn + bankroll - total_deposited,
        'withdrawals_log': pd.DataFrame(withdrawals_log),
        'deposits_log': pd.DataFrame(deposits_log),
    }

print('Motor de simulação pronto.')

## 4. Executar Simulações

In [ ]:
results = {}

for name, cfg in STRATEGIES.items():
    print(f'\nSimulando estratégia {name}...')
    r = simulate_strategy(
        mults=mults, streaks=streaks, hours=hours,
        weekdays=weekdays, dates=dates,
        green_slots=GREEN_SLOTS,
        pattern=cfg['pattern'], divisor=cfg['divisor'],
        trigger=TRIGGER, target=TARGET,
        bankroll_inicial=BANKROLL_INICIAL,
        meta_saque=META_SAQUE,
    )
    results[name] = r
    
    print(f'  Trades:        {r["total_trades"]:,}')
    print(f'  Win Rate:      {r["win_rate"]*100:.2f}%')
    print(f'  Saques:        {r["n_withdrawals"]:,} (R${r["total_withdrawn"]:,.2f})')
    print(f'  Depósitos:     {r["n_deposits"]:,} (R${r["total_deposited"]:,.2f})')
    print(f'  Banca final:   R${r["final_bankroll"]:,.2f}')
    print(f'  LUCRO LÍQUIDO: R${r["net_profit"]:+,.2f}')

## 5. Comparação das Estratégias

In [ ]:
print('=' * 70)
print('COMPARAÇÃO DAS ESTRATÉGIAS')
print('=' * 70)
print(f'{"":>25} {"1-2-4":>20} {"1-2":>20}')
print('-' * 70)

r1 = results['1-2-4']
r2 = results['1-2']

rows = [
    ('Total de Trades', f'{r1["total_trades"]:,}', f'{r2["total_trades"]:,}'),
    ('Win Rate', f'{r1["win_rate"]*100:.2f}%', f'{r2["win_rate"]*100:.2f}%'),
    ('Wins / Losses', f'{r1["wins"]:,} / {r1["losses"]:,}', f'{r2["wins"]:,} / {r2["losses"]:,}'),
    ('', '', ''),
    ('Total Depositado', f'R${r1["total_deposited"]:,.2f}', f'R${r2["total_deposited"]:,.2f}'),
    ('Total Sacado', f'R${r1["total_withdrawn"]:,.2f}', f'R${r2["total_withdrawn"]:,.2f}'),
    ('Nº Saques', f'{r1["n_withdrawals"]:,}', f'{r2["n_withdrawals"]:,}'),
    ('Nº Depósitos', f'{r1["n_deposits"]:,}', f'{r2["n_deposits"]:,}'),
    ('Banca Final', f'R${r1["final_bankroll"]:,.2f}', f'R${r2["final_bankroll"]:,.2f}'),
    ('', '', ''),
    ('LUCRO LÍQUIDO', f'R${r1["net_profit"]:+,.2f}', f'R${r2["net_profit"]:+,.2f}'),
    ('ROI (lucro/depositado)', f'{r1["net_profit"]/r1["total_deposited"]*100:+.1f}%', f'{r2["net_profit"]/r2["total_deposited"]*100:+.1f}%'),
]

for label, v1, v2 in rows:
    print(f'{label:>25} {v1:>20} {v2:>20}')

# Período
days = (df['date'].max() - df['date'].min()).days
months = days / 30.44
print(f'\nPeríodo: {days:,} dias ({months:.0f} meses)')
print(f'Lucro mensal médio (1-2-4): R${r1["net_profit"]/months:,.2f}/mês')
print(f'Lucro mensal médio (1-2):   R${r2["net_profit"]/months:,.2f}/mês')

## 6. Curvas de Equity (Lucro Líquido Acumulado)

In [ ]:
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Lucro Líquido Acumulado', 'Saldo da Banca'),
    shared_xaxes=False,
)

for name, color in [('1-2-4', GREEN), ('1-2', CYAN)]:
    r = results[name]
    trades_df = r['trades_df']
    if len(trades_df) == 0:
        continue
    
    fig.add_trace(go.Scatter(
        x=trades_df['date'], y=r['equity_curve'],
        mode='lines', name=f'{name} (lucro)',
        line=dict(color=color, width=1.5),
    ), row=1, col=1)
    
    fig.add_trace(go.Scatter(
        x=trades_df['date'], y=r['bankroll_curve'],
        mode='lines', name=f'{name} (banca)',
        line=dict(color=color, width=1, dash='dot'),
    ), row=2, col=1)

fig.add_hline(y=0, line_dash='dash', line_color=RED, opacity=0.5, row=1, col=1)
fig.add_hline(y=BANKROLL_INICIAL, line_dash='dash', line_color=YELLOW, opacity=0.5, row=2, col=1,
              annotation_text='Banca base')

fig.update_layout(
    height=800,
    title_text='Simulação Realista: 1-2-4 vs 1-2 (Horários Verdes)',
)
fig.update_yaxes(title_text='R$', row=1, col=1)
fig.update_yaxes(title_text='R$', row=2, col=1)
fig.show()

## 7. Timeline de Saques e Depósitos

In [ ]:
for name in ['1-2-4', '1-2']:
    r = results[name]
    wl = r['withdrawals_log']
    dl = r['deposits_log']
    
    fig = go.Figure()
    
    if len(wl) > 0:
        wl['cumulative'] = wl['amount'].cumsum()
        fig.add_trace(go.Scatter(
            x=wl['date'], y=wl['cumulative'],
            mode='lines', name='Saques acumulados',
            line=dict(color=GREEN, width=2),
            fill='tozeroy', fillcolor='rgba(0,255,136,0.08)',
        ))
    
    if len(dl) > 0:
        dl['cumulative'] = dl['amount'].cumsum()
        fig.add_trace(go.Scatter(
            x=dl['date'], y=dl['cumulative'],
            mode='lines', name='Depósitos acumulados',
            line=dict(color=RED, width=2),
            fill='tozeroy', fillcolor='rgba(255,51,102,0.08)',
        ))
    
    fig.update_layout(
        title=f'Estratégia {name}: Saques vs Depósitos ao Longo do Tempo',
        xaxis_title='', yaxis_title='R$ Acumulado',
        height=450,
    )
    fig.show()
    
    print(f'\n--- {name} ---')
    if len(wl) > 0:
        print(f'Primeiro saque: {pd.Timestamp(wl.iloc[0]["date"]).strftime("%Y-%m-%d %H:%M")}')
        print(f'Último saque:   {pd.Timestamp(wl.iloc[-1]["date"]).strftime("%Y-%m-%d %H:%M")}')
        print(f'Frequência:     1 saque a cada {len(r["trades_df"])/len(wl):.0f} trades')
    if len(dl) > 0:
        print(f'Reposições:     {len(dl)} (1 a cada {len(r["trades_df"])/len(dl):.0f} trades)')

## 8. Análise Mensal

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Estratégia 1-2-4', 'Estratégia 1-2'),
)

for col, name in enumerate(['1-2-4', '1-2'], 1):
    r = results[name]
    trades_df = r['trades_df'].copy()
    if len(trades_df) == 0:
        continue
    
    trades_df['ano_mes'] = pd.to_datetime(trades_df['date']).dt.strftime('%Y-%m')
    monthly = trades_df.groupby('ano_mes').agg(
        pnl=('pnl', 'sum'),
        trades=('pnl', 'count'),
        wr=('hit', 'mean'),
    ).reset_index()
    
    fig.add_trace(go.Bar(
        x=monthly['ano_mes'], y=monthly['pnl'],
        marker_color=[GREEN if p > 0 else RED for p in monthly['pnl']],
        name=name,
        text=[f'R${p:+,.0f}' for p in monthly['pnl']],
        textposition='outside',
        textfont=dict(size=8),
    ), row=1, col=col)

fig.update_layout(height=500, title_text='PnL Mensal por Estratégia', showlegend=False)
fig.update_yaxes(title_text='R$')
fig.show()

# Tabela mensal
for name in ['1-2-4', '1-2']:
    r = results[name]
    trades_df = r['trades_df'].copy()
    if len(trades_df) == 0:
        continue
    trades_df['ano_mes'] = pd.to_datetime(trades_df['date']).dt.strftime('%Y-%m')
    monthly = trades_df.groupby('ano_mes').agg(
        pnl=('pnl', 'sum'),
        trades=('pnl', 'count'),
        wr=('hit', 'mean'),
    ).reset_index()
    
    pos = (monthly['pnl'] > 0).sum()
    print(f'\n--- {name}: {pos}/{len(monthly)} meses positivos ({pos/len(monthly)*100:.0f}%) ---')
    print(f'{"Mês":>10} {"PnL":>12} {"Trades":>8} {"WR":>8}')
    print('-' * 42)
    for _, row in monthly.iterrows():
        print(f'{row["ano_mes"]:>10} R${row["pnl"]:>+10,.2f} {int(row["trades"]):>8} {row["wr"]*100:>7.1f}%')

## 9. Resumo Final

In [ ]:
days = (df['date'].max() - df['date'].min()).days
months = days / 30.44

print('=' * 60)
print('RESULTADO FINAL DA SIMULAÇÃO')
print('=' * 60)

for name in ['1-2-4', '1-2']:
    r = results[name]
    cfg = STRATEGIES[name]
    
    print(f'\n--- Estratégia {name} (unit = banca/{cfg["divisor"]}) ---')
    print(f'  Banca inicial:     R${BANKROLL_INICIAL:,.2f}')
    print(f'  Total depositado:  R${r["total_deposited"]:,.2f} ({r["n_deposits"]} depósitos)')
    print(f'  Total sacado:      R${r["total_withdrawn"]:,.2f} ({r["n_withdrawals"]} saques)')
    print(f'  Banca final:       R${r["final_bankroll"]:,.2f}')
    print(f'  ')
    print(f'  LUCRO LÍQUIDO:     R${r["net_profit"]:+,.2f}')
    print(f'  Lucro/mês:         R${r["net_profit"]/months:+,.2f}')
    print(f'  ROI:               {r["net_profit"]/r["total_deposited"]*100:+.1f}%')
    print(f'  ')
    print(f'  Trades:            {r["total_trades"]:,}')
    print(f'  Win Rate:          {r["win_rate"]*100:.2f}%')

# Qual é melhor?
better = '1-2-4' if results['1-2-4']['net_profit'] > results['1-2']['net_profit'] else '1-2'
diff = abs(results['1-2-4']['net_profit'] - results['1-2']['net_profit'])
print(f'\n{"="*60}')
print(f'VENCEDORA: Estratégia {better} (R${diff:,.2f} a mais de lucro)')
print(f'{"="*60}')